In [71]:
# 1-1
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/7_1_1.csv')
# print(df.info())
# print(df.head())
df['신고시각'] = df['신고시각'].astype(str)
df['처리시각'] = df['처리시각'].astype(str)
df['처리일자'] = df['처리일자'].astype(str)
df['신고일자'] = df['신고일자'].astype(str)

df['신고시각'] = df['신고시각'].str.zfill(6)
df['처리시각'] = df['처리시각'].str.zfill(6)

df['신고일시'] = df['신고일자'] + df['신고시각']
df['처리일시'] = df['처리일자'] + df['처리시각']

df['신고일시'] = pd.to_datetime(df['신고일시'], format='%Y%m%d%H%M%S')
df['처리일시'] = pd.to_datetime(df['처리일시'], format='%Y%m%d%H%M%S')
df['처리시간'] = df['처리일시'] - df['신고일시']
df.groupby('공장명')['처리시간'].mean().sort_values(ascending=True)

공장명
공장B      -20 days +22:29:36.062500
공장C   -19 days +12:01:29.073684211
공장D      3 days 14:19:17.888888888
공장A     10 days 10:36:08.504587156
공장E     10 days 16:52:31.381818181
Name: 처리시간, dtype: timedelta64[ns]

In [101]:
# 1-2
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/7_1_2.csv')
# print(df.info())
# print(df.head())

df['구'] = df['STATION_ADDR1'].str.split(' ').str[1]
# df['이동거리'] = np.abs(df['STATION_LATITUDE'] - df['STATION_LONGITUDE'])
df1 = df[(df['구'] == '마포구') | (df['구'] == '성동구')]
df1.groupby('구')['dist'].mean().round(2)

구
마포구    3045.88
성동구    3202.31
Name: dist, dtype: float64

In [102]:
# 1-3
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/7_1_3.csv')
# print(df.info())
# print(df.head())

df['합계'] = df['제품A'] + df['제품B'] + df['제품C'] + df['제품D'] + df['제품E']
df['년'] = df['기간'].str.split('_').str[0]
df['월'] = df['기간'].str.split('_').str[1].str.replace('월', '').astype(int)
df['분기'] = df['월'].apply(lambda x: 1 if x in [1,2,3] else (2 if x in [4,5,6] else (3 if x in [7,8,9] else 4)))
df.groupby(['년', '분기'])['합계'].mean().sort_values(ascending=False)

년      분기
2018년  3     3226.666667
2019년  3     3118.333333
2018년  1     3064.000000
       4     2959.000000
2019년  1     2914.000000
       4     2456.000000
2018년  2     2270.000000
2019년  2     1812.333333
Name: 합계, dtype: float64

In [35]:
# 2
import numpy as np
import pandas as pd
train_df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/7_2_train.csv')
test_df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/7_2_test.csv')

print(train_df.info())
print(train_df.head())
# print(test_df.info())

train_y = train_df['Target']
train_X = train_df.drop(columns=['Target', 'level_0', 'ID'])
# print(train_X.info())
test_y = test_df['Target']
test_X = test_df.drop(columns=['Target', 'level_0', 'ID'])

from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import get_scorer, f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
# f1_macro

train_X, valid_X, train_y, valid_y = train_test_split(train_X, train_y, test_size=0.3)
# print(train_X.shape, train_y.shape, valid_X.shape, valid_y.shape)

# train_y.value_counts()
train_y = train_y.map({'Dropout': 0, 'Enrolled': 1, 'Graduate': 2})
valid_y = valid_y.map({'Dropout': 0, 'Enrolled': 1, 'Graduate': 2})
test_y = test_y.map({'Dropout': 0, 'Enrolled': 1, 'Graduate': 2})

obj = train_X.select_dtypes('object').columns.tolist()
num = train_X.select_dtypes('number').columns.tolist()

train_X[obj] = train_X[obj].replace('_', '', regex=True)
valid_X[obj] = valid_X[obj].replace('_', '', regex=True)
test_X[obj] = test_X[obj].replace('_', '', regex=True)
# train_X.head()

ordinalencoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
train_X_encoder = ordinalencoder.fit_transform(train_X[obj])
valid_X_encoder = ordinalencoder.transform(valid_X[obj])
test_X_encoder = ordinalencoder.transform(test_X[obj])

train_X = np.concatenate([train_X_encoder, train_X[num]], axis=1)
valid_X = np.concatenate([valid_X_encoder, valid_X[num]], axis=1)
test_X = np.concatenate([test_X_encoder, test_X[num]], axis=1)

rf = RandomForestClassifier()
rf.fit(train_X, train_y)
pred1 = rf.predict(valid_X)
print(f1_score(valid_y, pred1, average='macro'))

# pred2 = rf.predict(test_X)
# pd.DataFrame(pred2, columns=['pred']).to_csv('result.csv', index=False)

params={
    'max_depth': [10,20,30],
    'min_samples_split': [2,5,10]
}
cv = GridSearchCV(estimator=rf, param_grid=params, scoring='f1_macro', cv=3)
cv.fit(train_X, train_y)
pred3 = cv.predict(valid_X)
print(f1_score(valid_y, pred3, average='macro'))

pred4 = cv.predict(test_X)
pd.DataFrame(pred4, columns=['pred']).to_csv('result.csv', index=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 309 entries, 0 to 308
Data columns (total 19 columns):
 #   Column                                  Non-Null Count  Dtype 
---  ------                                  --------------  ----- 
 0   level_0                                 309 non-null    int64 
 1   ID                                      309 non-null    int64 
 2   Marital Status                          309 non-null    object
 3   Application mode                        309 non-null    object
 4   Course                                  309 non-null    object
 5   Daytime/evening attendance              309 non-null    object
 6   Previous qualification                  309 non-null    object
 7   Nacionality                             309 non-null    object
 8   Mother's qualification                  309 non-null    object
 9   Father's qualification                  309 non-null    object
 10  Mother's occupation                     309 non-null    object
 11  Father

In [94]:
# 3-1
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/7_3_1.csv')
print(df.info())
print(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   before  50 non-null     float64
 1   after   50 non-null     float64
dtypes: float64(2)
memory usage: 932.0 bytes
None
      before      after
0  87.640523  88.163191
1  74.001572  80.936085
2  79.787380  82.233354
3  92.408932  91.505771
4  88.675580  93.534669


In [113]:
# 3-2
import numpy as np
import pandas as pd
train_df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/7_3_2_train.csv')
test_df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/7_3_2_test.csv')

# print(train_df.info())
# print(train_df.head())
# print(test_df.info())

import statsmodels.api as sm
import statsmodels.formula.api as smf

model = smf.glm("purchase ~ age + income + family_members", data=train_df, family=sm.families.Binomial()).fit()
print(np.exp(model.params['income']).round(3))
print(model.deviance.round(2))

test_y = test_df['purchase']
test_X = test_df.drop(columns=['purchase'])
pred1 = model.predict(test_X)
pred1 = (pred1 > 0.5).astype(int)
from sklearn.metrics import accuracy_score
1 - accuracy_score(test_y, pred1)

1.0
108.19


0.35